## Introducción

En esta notebook se evaluará **TextBlob**, una biblioteca de procesamiento de lenguaje natural (NLP) que incorpora un modelo preentrenado para análisis de sentimiento en idioma inglés.

A diferencia de los modelos clásicos desarrollados en la notebook anterior, TextBlob no requiere una etapa de entrenamiento sobre el conjunto de datos del proyecto, ya que utiliza un analizador de sentimiento previamente entrenado. Esto permitirá comparar el desempeño de un modelo preentrenado frente a los modelos de Machine Learning entrenados específicamente con el dataset de tweets.

La evaluación se realizará utilizando el conjunto de prueba `testdata.manual.2009.06.14.csv`, el cual contiene tweets etiquetados manualmente como **negativos**, **neutrales** y **positivos**. Los resultados obtenidos se analizarán mediante distintas métricas de clasificación y servirán como base para la comparación final entre los distintos enfoques utilizados en este trabajo.


In [1]:
import pandas as pd

from textblob import TextBlob

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix,
    classification_report
)

In [2]:
#Carga del dataset

column_names = [
    "polarity",
    "id",
    "date",
    "query",
    "user",
    "text"
]

df_test = pd.read_csv(
    "../data/raw/testdata.manual.2009.06.14.csv",
    encoding="latin-1",
    header=None,
    names=column_names,
    usecols=["polarity", "text"]
)

In [4]:
#validaciones

df_test.shape

(498, 2)

In [5]:
df_test.head()

,polarity,text
0,4,@stellargirl I loooooooovvvvvveee my Kindle2. ...
1,4,Reading my kindle2... Love it... Lee childs i...
2,4,"Ok, first assesment of the #kindle2 ...it fuck..."
3,4,@kenburbary You'll love your Kindle2. I've had...
4,4,@mikefish Fair enough. But i have the Kindle2...


In [6]:
df_test.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 498 entries, 0 to 497
Data columns (total 2 columns):
 #   Column    Non-Null Count  Dtype 
---  ------    --------------  ----- 
 0   polarity  498 non-null    int64 
 1   text      498 non-null    object
dtypes: int64(1), object(1)
memory usage: 7.9+ KB


In [7]:
df_test["polarity"].value_counts()

polarity
4    182
0    177
2    139
Name: count, dtype: int64

## Preprocesamiento del conjunto de prueba

Para la evaluación de TextBlob se aplicará únicamente la etapa de normalización del texto desarrollada en la Notebook 2. Se realizarán la conversión a minúsculas, la eliminación de URLs y menciones, la conversión de entidades HTML, la conservación del contenido de los hashtags y la normalización de espacios.

No se aplicará lematización ni eliminación de stopwords, ya que TextBlob es un modelo preentrenado diseñado para trabajar sobre texto en lenguaje natural. Mantener la estructura original del texto permite evaluar el modelo en las condiciones para las cuales fue concebido, evitando modificar información lingüística que podría influir en la estimación del sentimiento.


In [8]:
import re
import string
import html

# Función para eliminar URLs
def remove_urls(text):
    return re.sub(r"http\S+|www\S+", "", text)

# Función para eliminar menciones
def remove_mentions(text):
    return re.sub(r"@\w+", "", text)

# Función para conservar el texto de los hashtags eliminando solo '#'
def process_hashtags(text):
    return text.replace("#", "")

# Función para convertir entidades HTML
def decode_html(text):
    return html.unescape(text)

# Función para normalizar espacios
def normalize_spaces(text):
    return re.sub(r"\s+", " ", text).strip()

# Función principal de normalización
def clean_text(text):

    # Conversión a minúsculas
    text = text.lower()

    # Eliminación de URLs
    text = remove_urls(text)

    # Eliminación de menciones
    text = remove_mentions(text)

    # Conservación del contenido de los hashtags
    text = process_hashtags(text)

    # Conversión de entidades HTML
    text = decode_html(text)

    # Normalización de espacios
    text = normalize_spaces(text)

    return text

In [9]:
# Aplicación del proceso de normalización

df_test["normalized_text"] = df_test["text"].apply(clean_text)

In [10]:
df_test[["text", "normalized_text"]].head()

,text,normalized_text
0,@stellargirl I loooooooovvvvvveee my Kindle2. ...,i loooooooovvvvvveee my kindle2. not that the ...
1,Reading my kindle2... Love it... Lee childs i...,reading my kindle2... love it... lee childs is...
2,"Ok, first assesment of the #kindle2 ...it fuck...","ok, first assesment of the kindle2 ...it fucki..."
3,@kenburbary You'll love your Kindle2. I've had...,you'll love your kindle2. i've had mine for a ...
4,@mikefish Fair enough. But i have the Kindle2...,fair enough. but i have the kindle2 and i thin...


In [11]:
df_test["normalized_text"].sample(5, random_state=42)

487    getting ready to test out some burger receipes...
73     back when i worked for nike we had one fav wor...
231    yay, glad you got the phone! still, damn you, ...
175                          waiting in line at safeway.
237                           reading on my new kindle2!
Name: normalized_text, dtype: object

## Conversión de la polaridad a clases de sentimiento

TextBlob no devuelve directamente una categoría de sentimiento, sino un valor de **polaridad** comprendido entre **-1** y **1**. Los valores negativos representan sentimientos desfavorables, los valores positivos representan sentimientos favorables y los valores cercanos a cero indican neutralidad.

Para poder comparar las predicciones de TextBlob con las etiquetas del dataset de evaluación, es necesario transformar esta polaridad en las tres clases utilizadas por el conjunto de datos: **0 (negativo)**, **2 (neutral)** y **4 (positivo)**.

En una primera aproximación se utilizará una regla de decisión simple: las polaridades menores que cero se clasificarán como **negativas**, las iguales a cero como **neutrales** y las mayores que cero como **positivas**. Esta estrategia permitirá evaluar el comportamiento del modelo preentrenado utilizando los mismos criterios de clasificación definidos en el dataset de referencia.


In [12]:
# Función para convertir la polaridad en clases

def predict_sentiment(text):

    polarity = TextBlob(text).sentiment.polarity

    if polarity < 0:
        return 0

    elif polarity == 0:
        return 2

    else:
        return 4

In [13]:
# Predicción del sentimiento

df_test["prediction"] = df_test["normalized_text"].apply(
    predict_sentiment
)

In [14]:
df_test["prediction"].value_counts()

prediction
4    233
2    157
0    108
Name: count, dtype: int64

### Distribución de las predicciones

Como primera validación del modelo se analizó la distribución de las clases predichas por TextBlob.

Se observa una mayor cantidad de predicciones positivas respecto de la distribución original del conjunto de datos, mientras que la cantidad de predicciones negativas resulta inferior. Por otro lado, la cantidad de predicciones neutrales es cercana a la cantidad de ejemplos presentes en el dataset.

Esta comparación constituye una primera aproximación al comportamiento del modelo y será complementada mediante las métricas de clasificación y el análisis de la matriz de confusión.


In [15]:
# Etiquetas reales y predichas

y_true = df_test["polarity"]

y_pred = df_test["prediction"]

In [16]:
print(classification_report(y_true, y_pred))

              precision    recall  f1-score   support

           0       0.80      0.49      0.60       177
           2       0.60      0.68      0.64       139
           4       0.61      0.78      0.68       182

    accuracy                           0.65       498
   macro avg       0.67      0.65      0.64       498
weighted avg       0.67      0.65      0.64       498



In [17]:
cm = confusion_matrix(y_true, y_pred)

cm

array([[ 86,  37,  54],
       [  8,  94,  37],
       [ 14,  26, 142]])

### Análisis de resultados

TextBlob obtuvo una **Accuracy del 65 %** sobre el conjunto de prueba, alcanzando un desempeño diferente según la clase analizada.

La clase **positiva** presentó el mejor nivel de recuperación, evidenciado por un *Recall* de 0.78, mientras que la clase **negativa** obtuvo la mayor *Precision* (0.80), aunque con un *Recall* considerablemente menor. Esto indica que el modelo identifica correctamente los sentimientos negativos cuando los predice, pero deja sin detectar una parte importante de los ejemplos pertenecientes a esa categoría.

En cuanto a la clase **neutral**, TextBlob logró un comportamiento equilibrado, obteniendo valores de *Precision* y *Recall* cercanos a 0.60 y 0.68, respectivamente. Considerando que la identificación de sentimientos neutrales suele representar una tarea más compleja, estos resultados pueden considerarse satisfactorios para un modelo preentrenado de propósito general.

La matriz de confusión muestra una tendencia del modelo a clasificar una parte de los tweets negativos como positivos o neutrales, mientras que la identificación de los tweets positivos presenta un mejor desempeño. Este comportamiento resulta esperable, dado que TextBlob no fue entrenado específicamente sobre el conjunto de datos utilizado en este trabajo, sino que emplea un analizador de sentimiento previamente desarrollado para aplicaciones generales.
